# Introduction

Sometimes, even though you might not see the exact term you’re looking for, the document might contain similar words or phrases that have the same meaning or context but don’t have the exact same form (such as differences in spelling).

Traditional NLP search approaches have relied on using exact forms to search for words or phrases in a particular document. But this fails at finding words based on semantic or contextual meaning.

To solve this, semantic matching comes into play. It’s an advanced way of searching that takes advantage of traditional search methods while also focusing more on locating or matching words or phrases based on their meaning or context (rather than solely on their exact form).


In [18]:
# import library
import PyPDF2

# use PDFreader from PyPDF to read pdf content.

pdf_reader = PyPDF2.PdfReader('Relationships_Education_RSE_and_Health_Education.pdf')

# join all the content in the pdf pages together and lowercase the letters
pdf_document = " ".join([page.extract_text().lower() for page in pdf_reader.pages])

# check if the string 'Rachit Singh' is in the documnet [return false]
"birth control" in pdf_document

False

<h2>What is semantic Matching?</h2>

Semantic Matching is a technique used to determine if two elements have the same meaning. An element can be a word, phrase, sentence, document, or even a corpus. It refers to Matching elements based on meaning or context and not just matching based on exact form. 

<h2> What is Word Embedding? </h2>

Word embedding is an advanced text representation technique used to represent words in a lower-dimensional vector representation. This vector representation captures inter-word semantic and syntactic information. This means that words that have similar meanings - even though they might be spelled differently - will have close to similar vector representation. 

<h2> What does Lower-Dimensional Vector representation </h2>

In NLP, traditional ways of representing text in a way machines can understand(that is numerical vector representations) are Bag of Words, Term-Frequency and Inverse Document Frequency (TF-IDF), and One-hot encoding. But these techniques usually generate high dimensions (usually the size of the vocbulary) for a particular word representation and are sparse (meaning there will be lots of zeros).

So for example, if a word is to be represented as a numerical vector and the document or corpus the word belongs to has 10,000 vocabularies, the size of the dimension of that word would be 10,000 (making it high).

The disadvantages of these techniques are high dimensions, sparsity, and thei non-capability in capturing semantic information. So, advancements in NLP led to the development of word embedding techniques that simply create lower (also known as more dense) vector representations of words and can capture inter-word semantic information. 

Word embedding is the holy grail in NLP and language technology, serving as the foundation for advanced language representation models such as GPT (Generative Pre-trained Transformer)

There is also sentence embedding that represents sentences in a lower-dimension vector representation.

In [19]:
from sentence_transformers import SentenceTransformer 
from sentence_transformers.util import cos_sim
# import torch

model = SentenceTransformer("all-mpnet-base-v2")

In [20]:
man_vector = model.encode('man')
woman_vector = model.encode('woman')
cat_vector = model.encode('cat')

# get the similarity between man and woman 
smiliarity = cos_sim(man_vector,woman_vector)

# get the similarity between man and cat
cat_similarity = cos_sim(man_vector, cat_vector)

print('The similarity between man vector and woman vector: ',smiliarity, "\n" )
print('The similarity between Man vector and Cat Vector: ', cat_similarity)

The similarity between man vector and woman vector:  tensor([[0.3501]]) 

The similarity between Man vector and Cat Vector:  tensor([[0.2553]])


In [21]:
from keybert import KeyBERT
#T his is a library that allows us to get meaningful keywords (words or phrases) from a particular document in a minimal way.
# initialize model 
keybert_model = KeyBERT()

# extract all keywords (single word and 2 word phrase) from the pdf
all_keywords = keybert_model.extract_keywords(docs=pdf_document, top_n=-1, keyphrase_ngram_range=(1,2))
# print length of keywords extracted
print(len(all_keywords))
#show the first 5 keywords
print(all_keywords[:5])

8669
[('education guidance', 0.5954), ('schools guidance', 0.5542), ('education policies', 0.5405), ('sex education', 0.5228), ('education safeguarding', 0.5001)]


In [22]:
# remove score form each keyword
all_keywords = [keyword[0] for keyword in all_keywords]
all_keywords[:5]

['education guidance',
 'schools guidance',
 'education policies',
 'sex education',
 'education safeguarding']

In [23]:
# initialize sentence transformer with the 'all-mpnet-base-v2' model
model = SentenceTransformer('all-mpnet-base-v2')

# get the embedding of the 'birth control' phrase
birth_control_embedding = model.encode('birth control')

# get the embedidng of all the keywords in the document
keywords_embedding = model.encode(all_keywords)

After getting the embedding of the phrase and the keywords, the next step is to get the similarity score of the phrase and the keywords. This will help us know which keyword in the document is highly similar to the phrase.

In [25]:
# calculate the cosine similarity of the birth control word and each word in the document
cosine_similarity_result = cos_sim(birth_control_embedding,keywords_embedding)

# print the shape (equal to the number of keywords)
print(cosine_similarity_result.shape)

# show the top 5 similarites
print(cosine_similarity_result)

torch.Size([1, 8669])
tensor([[0.1252, 0.1084, 0.1402,  ..., 0.0990, 0.1710, 0.0508]])


In [27]:
index = cosine_similarity_result.argmax()
print(index)
# It tells us that the keyword with the highest similarity to the Birth Control phrase is at index 1490.

tensor(1490)


In [28]:
print(all_keywords[index])

contraceptive


After examining it, we found that "contraceptive" was the word with the highest similarity, which makes sense because "birth control" and "contraceptive" mean the same thing. This demonstrates the elegance of semantic matching in finding similar words.

# Let’s Also Explore Top 5 Keywords in the PDF that Match with the Phrase “Birth Control”
To do that, we can use the topk() method to get the top 5 indices. Then we can then loop through these indices to get the actual keywords:

In [29]:
top_5_indices = cosine_similarity_result.topk(5)[1].tolist()[0]

print(top_5_indices)

[1490, 1972, 871, 1199, 1944]


In [30]:
# get top 5 keywords
top_5_keywords = [all_keywords[index] for index in top_5_indices]
print(top_5_keywords)

['contraceptive', 'contraception', 'contraceptive choices', 'range contraceptive', 'cover contraception']


There, we can see that the top five results relate to contraception and contraceptives. This demonstrates that semantic matching is an effective way to find related elements in a document.